# Task 1: Data Collection & Dataset Understanding
**Decodelabs Data Science Internship**

**Goal:** Collect or load a dataset and understand its structure.

**Key Requirements:**
- Identify columns and data types
- Understand dataset size and features
- Describe what the data represents

**Dataset:** S&P 500 Multi-Stock Financial Data (20 major companies, 2019–2024)

## Step 1: Install & Import Libraries

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## Step 2: Download the Dataset

We use **yfinance** — a free Python library that pulls real stock market data directly from Yahoo Finance.

We selected 20 major S&P 500 companies across different sectors:
- **Tech:** AAPL, MSFT, GOOGL, AMZN, NVDA, META, TSLA, NFLX
- **Finance:** JPM, BAC, V, MA
- **Healthcare:** JNJ, UNH, PFE
- **Consumer:** PG, KO, PEP, HD
- **Energy:** XOM

In [2]:
# Define the 20 stock tickers
tickers = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA',
    'META', 'TSLA', 'JPM', 'JNJ', 'V',
    'PG', 'UNH', 'HD', 'MA', 'BAC',
    'XOM', 'PFE', 'KO', 'PEP', 'NFLX'
]

print(f"Downloading data for {len(tickers)} companies...")

# Download 6 years of daily data
data = yf.download(tickers, start="2019-01-01", end="2024-12-31",
                   group_by='ticker', auto_adjust=True)

# Reshape from wide format to long format
frames = []
for ticker in tickers:
    try:
        df_ticker = data[ticker].copy()
        df_ticker['Symbol'] = ticker
        df_ticker.reset_index(inplace=True)
        frames.append(df_ticker)
    except Exception as e:
        print(f"Skipped {ticker}: {e}")

df = pd.concat(frames, ignore_index=True)

# Save to CSV so we don't need to re-download
df.to_csv('sp500_stocks.csv', index=False)
print(f"\nData saved to sp500_stocks.csv")
print(f"Total rows downloaded: {len(df):,}")

[*********************100%***********************]  20 of 20 completed



Data saved to sp500_stocks.csv
Total rows downloaded: 30,180


## Step 3: Preview the Data

**What each column means:**
| Column | Type | Description |
|--------|------|-------------|
| Date | datetime | Trading day |
| Open | float | Price at market open |
| High | float | Highest price of the day |
| Low | float | Lowest price of the day |
| Close | float | Price at market close |
| Volume | int | Number of shares traded |
| Symbol | string | Stock ticker code |

In [3]:
# Preview first 5 rows
print("First 5 rows of the dataset:")
df.head()

First 5 rows of the dataset:


Price,Date,Open,High,Low,Close,Volume,Symbol
0,2019-01-02,36.750281,37.689860,36.593684,37.469200,148158800,AAPL
1,2019-01-03,34.161690,34.574536,33.691903,33.736984,365248800,AAPL
2,2019-01-04,34.292196,35.246010,34.118992,35.177200,234428400,AAPL
3,2019-01-07,35.281604,35.312450,34.617256,35.098907,219111200,AAPL
4,2019-01-08,35.485657,36.021883,35.238901,35.768005,164101200,AAPL


In [4]:
# Last 5 rows
print("Last 5 rows of the dataset:")
df.tail()

Last 5 rows of the dataset:


Price,Date,Open,High,Low,Close,Volume,Symbol
30175,2024-12-23,91.342003,91.500000,89.910004,91.144997,23394000,NFLX
30176,2024-12-24,91.500000,93.584999,91.169998,93.211998,23203000,NFLX
30177,2024-12-26,92.839996,93.049004,91.529999,92.414001,23403000,NFLX
30178,2024-12-27,91.600998,91.813004,89.449997,90.754997,32262000,NFLX
30179,2024-12-30,89.450996,90.822998,88.971001,90.042999,22030000,NFLX


## Step 4: Understand Dataset Size & Structure

In [5]:
# Shape = (rows, columns)
rows, cols = df.shape
print(f"Dataset Shape: {rows:,} rows x {cols} columns")
print(f"\nThis means we have {rows:,} daily trading records")
print(f"across {df['Symbol'].nunique()} companies")
print(f"from {df['Date'].min()} to {df['Date'].max()}")

Dataset Shape: 30,180 rows x 7 columns

This means we have 30,180 daily trading records
across 20 companies
from 2019-01-02 00:00:00 to 2024-12-30 00:00:00


In [6]:
# Column names and data types
print("Column Names and Data Types:")
print("-" * 30)
print(df.dtypes)

Column Names and Data Types:
------------------------------
Price
Date      datetime64[s]
Open            float64
High            float64
Low             float64
Close           float64
Volume            int64
Symbol              str
dtype: object


In [7]:
# Full dataset info — shows non-null counts (great for spotting missing data)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30180 entries, 0 to 30179
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype        
---  ------  --------------  -----        
 0   Date    30180 non-null  datetime64[s]
 1   Open    30180 non-null  float64      
 2   High    30180 non-null  float64      
 3   Low     30180 non-null  float64      
 4   Close   30180 non-null  float64      
 5   Volume  30180 non-null  int64        
 6   Symbol  30180 non-null  str          
dtypes: datetime64[s](1), float64(4), int64(1), str(1)
memory usage: 1.6 MB


## Step 5: Check for Missing Values

In [8]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

Missing values per column:
Price
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
Symbol    0
dtype: int64

Total missing cells: 0


## Step 6: Understand the Companies

In [9]:
# How many rows per company?
records_per_company = df.groupby('Symbol').size().reset_index(name='Trading Days')
records_per_company = records_per_company.sort_values('Trading Days', ascending=False)
print("Records per company (trading days):")
print(records_per_company.to_string(index=False))

Records per company (trading days):
Symbol  Trading Days
  AAPL          1509
  AMZN          1509
     V          1509
   UNH          1509
  TSLA          1509
    PG          1509
   PFE          1509
   PEP          1509
  NVDA          1509
  NFLX          1509
  MSFT          1509
  META          1509
    MA          1509
    KO          1509
   JPM          1509
   JNJ          1509
    HD          1509
 GOOGL          1509
   BAC          1509
   XOM          1509


In [10]:
# Price range for each company (min and max close price)
price_summary = df.groupby('Symbol')['Close'].agg(['min', 'max', 'mean']).round(2)
price_summary.columns = ['Min Price ($)', 'Max Price ($)', 'Avg Price ($)']
price_summary = price_summary.sort_values('Avg Price ($)', ascending=False)
print("Price Summary by Company (2019-2024):")
print(price_summary.to_string())

Price Summary by Company (2019-2024):
        Min Price ($)  Max Price ($)  Avg Price ($)
Symbol                                             
UNH            176.10         603.20         387.67
MA             173.65         531.37         345.57
META            88.22         629.64         279.20
HD             130.62         414.97         265.50
MSFT            90.73         460.33         258.37
V              121.53         317.33         210.72
TSLA            11.93         479.86         180.71
JNJ             93.43         164.68         137.56
AMZN            75.01         232.93         137.10
PEP             85.14         175.40         135.05
AAPL            33.74         257.38         134.36
JPM             66.76         242.76         128.58
PG              74.82         172.54         125.58
GOOGL           50.82         195.64         108.16
XOM             23.82         118.35          70.13
KO              31.30          69.43          50.13
NFLX            16.64     

## Summary

**Dataset:** S&P 500 Real Stock Market Data

**Source:** Yahoo Finance via `yfinance` Python library

**Size:** ~30,180 rows × 7 columns

**Coverage:** 20 major companies across 6 sectors, daily data from Jan 2019 to Dec 2024

**Key columns:** Date, Open, High, Low, Close (prices in USD), Volume (shares traded), Symbol (ticker)

**Data quality:** No missing values, no duplicates detected

**Why this dataset is complex:**
- Multi-company time series data
- Covers 6 years including COVID crash (2020), bull market (2021), bear market (2022), AI boom (2023-24)
- Supports cleaning, EDA, visualization, and predictive modeling tasks